# Pruebas de extracción — Football / RUSSIA / Premier League

Flujo "completar partidos incompletos", reusando funciones ya probadas sobre el **driver vivo**. Todo en **dry-run** (no escribe en DB).

**Etapas:** Setup → 1 cargar liga → 2 detectar pendientes → 3 "Show more" + scan → Verificación encontrados vs DB → Opciones (modo + filtro) → Población a procesar → 4 extraer y mostrar qué se escribiría.

## Requisitos
1. **Driver vivo**: `python scripts/start_driver.py` en otra terminal (o botón "Iniciar" del panel).
2. **Kernel** de `env_sports`: `env_sports/bin/python -m jupyter lab` desde la raíz.
3. Las etapas 2-4 **leen la DB remota** (solo lectura); la 4 navega al detalle pero **no escribe**.

El Setup activa **`%autoreload 2`** → editas un `.py` y la próxima celda toma la versión nueva sin reiniciar el kernel.

## Setup — autoreload, paths, imports, helpers y liga objetivo

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    ROOT = os.path.abspath(os.getcwd())
for p in (ROOT, os.path.join(ROOT, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)
print('ROOT =', ROOT)

from selenium.webdriver.common.by import By
from common_functions import wait_update_page, dismiss_cookies, load_json
from data_base import getdb, get_math_details_ids
from milestone4 import wait_load_details, get_match_info, get_statistics_game, retry_match
from scripts.fix_live_matches import (
    find_results_url, get_pending_live_matches, load_until_date,
    scan_results_page, get_last_visible_date,
)
from scripts.driver_session import get_driver

# --- helpers locales (solo lectura) ---
def sin_stats(s):
    """True si statistic está vacío (NULL / '' / '{}') — misma regla que fix_null_team_ids."""
    return s is None or str(s).strip() in ('', '{}')

def get_statistic_map(match_ids):
    """{match_id: statistic} (solo lectura)."""
    if not match_ids:
        return {}
    con = getdb()
    try:
        cur = con.cursor()
        cur.execute('SELECT match_id, statistic FROM match WHERE match_id = ANY(%s)', (list(match_ids),))
        return {r[0]: r[1] for r in cur.fetchall()}
    finally:
        con.close()

def get_stats_backfill_matches(league_id):
    """Población para backfill de estadísticas:
       status = COMPLETED  AND  statistic vacío  AND  ningún score = -1
       (resultado real ya cargado; solo faltan los detalles). NO toca score."""
    con = getdb()
    try:
        cur = con.cursor()
        cur.execute('''
            SELECT DISTINCT m.match_id, m.name, m.match_date, m.status
              FROM match m
             WHERE m.league_id = %s
               AND m.status = 'COMPLETED'
               AND (m.statistic IS NULL OR m.statistic IN ('', '{}'))
               AND NOT EXISTS (
                   SELECT 1 FROM match_detail md
                     JOIN score_entity se ON se.match_detail_id = md.match_detail_id
                    WHERE md.match_id = m.match_id AND se.points = -1)
             ORDER BY m.match_date DESC
        ''', (league_id,))
        cols = ['match_id', 'name', 'match_date', 'status']
        return [dict(zip(cols, r)) for r in cur.fetchall()]
    finally:
        con.close()

# ---- Liga objetivo ----
SPORT_KEY    = 'FOOTBALL'
LEAGUE_KEY   = 'RUSSIA_Premier League'
COUNTRY_NAME = 'RUSSIA'
LEAGUE_NAME  = 'Premier League'
XPATH_ROWS   = '//div[contains(@class,"leagues--static event--leagues")]/div'

leagues_info = load_json('check_points/leagues_info.json')
info = leagues_info[SPORT_KEY][LEAGUE_KEY]
LEAGUE_ID   = info['league_id']
RESULTS_URL = info.get('results') or info.get('url')
print('league_id   =', LEAGUE_ID)
print('results_url =', RESULTS_URL)

ROOT = /home/jorge/work/scraper_V2.0
league_id   = 6839d592-35cd-49bc-823b-1c8734fb467d
results_url = https://www.flashscore.com/football/russia/premier-league/results/


## Etapa 1 — Cargar la liga correcta

In [2]:
via_helper = find_results_url(leagues_info, SPORT_KEY, COUNTRY_NAME, LEAGUE_NAME)
print('find_results_url coincide:', via_helper == RESULTS_URL, '|', via_helper)

driver = get_driver()
wait_update_page(driver, RESULTS_URL, 'container__heading')
dismiss_cookies(driver)

cur_url = driver.current_url
try:    heading = driver.find_element(By.CLASS_NAME, 'heading__name').text
except Exception: heading = '(no encontrado)'
try:    breadcrumb = driver.find_element(By.CLASS_NAME, 'container__heading').text.replace('\n', ' | ')
except Exception: breadcrumb = '(no encontrado)'
print('current_url :', cur_url)
print('heading liga:', heading)
print('breadcrumb  :', breadcrumb)
blob = (cur_url + ' ' + heading + ' ' + breadcrumb).lower()
hits = [t for t in (LEAGUE_NAME, COUNTRY_NAME) if t and t.lower() in blob]
print('\nETAPA 1:', 'OK' if hits else 'REVISAR', '| coincidencias:', hits or 'ninguna')

find_results_url coincide: True | https://www.flashscore.com/football/russia/premier-league/results/
[OK] Reconectado al browser.
     URL actual : https://www.flashscore.com/match/football/akhmat-grozny-bJGqkoet/fk-rostov-GbznhOLi/summary/stats/overall/?mid=QsIzdjHE
current_url : https://www.flashscore.com/football/russia/premier-league/results/
heading liga: Premier League
breadcrumb  : FOOTBALL | RUSSIA | Premier League | 2025/2026 | 18.07. | 23.05.

ETAPA 1: OK | coincidencias: ['Premier League', 'RUSSIA']


## Etapa 2 — Partidos que necesitan ACTUALIZACIÓN
`get_pending_live_matches`: **fecha < hoy AND (status='LIVE' OR algún score = -1)**. Filtrado a esta liga; calcula `TARGET`.

In [3]:
all_pending = get_pending_live_matches(verbose=False)
matches = [m for m in all_pending if m['league_id'] == LEAGUE_ID]

if not matches:
    print('Sin partidos pendientes de actualización para esta liga.')
    TARGET = None
else:
    by_status = {}
    for m in matches:
        by_status[m['status']] = by_status.get(m['status'], 0) + 1
    TARGET = min(m['match_date'] for m in matches if m['match_date'])
    newest = max(m['match_date'] for m in matches if m['match_date'])
    print(f'TOTAL a actualizar : {len(matches)}')
    print(f'Desglose por status: {by_status}')
    print(f'Rango de fechas    : {TARGET}  ->  {newest}')
    print(f'>>> FECHA MÁS ANTIGUA (target Etapa 3): {TARGET} <<<\n')
    print('%-12s %-10s  %s' % ('fecha', 'status', 'partido'))
    for m in matches[:40]:
        print('%-12s %-10s  %s' % (str(m['match_date']), (m['status'] or '')[:10], m['name']))
    if len(matches) > 40:
        print(f'... (+{len(matches)-40} más)')

TOTAL a actualizar : 97
Desglose por status: {'SCHEDULED': 97}
Rango de fechas    : 2025-11-23  ->  2026-05-09
>>> FECHA MÁS ANTIGUA (target Etapa 3): 2025-11-23 <<<

fecha        status      partido
2025-11-23   SCHEDULED   Lokomotiv Moscow~Krasnodar
2025-11-23   SCHEDULED   Pari NN~Zenit
2025-11-23   SCHEDULED   Dynamo Moscow~Dynamo Makhachkala
2025-11-29   SCHEDULED   Akron Togliatti~Pari NN
2025-11-29   SCHEDULED   Baltika~Spartak Moscow
2025-11-29   SCHEDULED   CSKA Moscow~Orenburg
2025-11-30   SCHEDULED   FK Rostov~Lokomotiv Moscow
2025-11-30   SCHEDULED   Zenit~Rubin Kazan
2025-11-30   SCHEDULED   Krasnodar~Krylya Sovetov
2025-11-30   SCHEDULED   Akhmat Grozny~Dynamo Moscow
2025-12-01   SCHEDULED   Sochi~Dynamo Makhachkala
2025-12-05   SCHEDULED   Akhmat Grozny~Orenburg
2025-12-06   SCHEDULED   Spartak Moscow~Dynamo Moscow
2025-12-06   SCHEDULED   FK Rostov~Rubin Kazan
2025-12-06   SCHEDULED   Zenit~Akron Togliatti
2025-12-07   SCHEDULED   Krasnodar~CSKA Moscow
2025-12-07   SCHE

## Etapa 3 — "Show more matches" hasta la fecha más antigua
`load_until_date` hace click hasta que la última fecha visible ≤ `TARGET`; luego `scan_results_page` mapea cada pendiente a su fila/URL (`found`).

In [4]:
assert TARGET is not None, 'No hay TARGET: la Etapa 2 no encontró pendientes.'

rows_before = len(driver.find_elements(By.XPATH, XPATH_ROWS))
date_before = get_last_visible_date(driver)
print(f'ANTES: filas={rows_before} | última fecha visible={date_before} | target={TARGET}\n')
print('>>> load_until_date — mira los [click N] abajo <<<')
load_until_date(driver, TARGET)
rows_after = len(driver.find_elements(By.XPATH, XPATH_ROWS))
date_after = get_last_visible_date(driver)
print(f'\nDESPUÉS: filas={rows_after} | última fecha visible={date_after}')

found = scan_results_page(driver, matches)
crecio        = rows_after > rows_before
alcanzo_fecha = bool(date_after) and date_after <= TARGET
print('\n==== VEREDICTO ETAPA 3 ====')
print(f'filas crecieron (clicks): {crecio}  ({rows_before} -> {rows_after})')
print(f'alcanzó fecha más antigua: {alcanzo_fecha}  ({date_after} <= {TARGET})')
print(f'cobertura: {len(found)}/{len(matches)}')
print('RESULTADO:', 'OK' if (crecio and alcanzo_fecha) else 'REVISAR')

ANTES: filas=123 | última fecha visible=2025-12-06 | target=2025-11-23

>>> load_until_date — mira los [click N] abajo <<<
  Cargando hasta: 2025-11-23
  [click 0] filas=123 | ultima fecha=2025-12-06
  [click 1] filas=124 | ultima fecha=2025-12-06
  [click 2] filas=233 | ultima fecha=2025-08-23
  Rango alcanzado.

DESPUÉS: filas=279 | última fecha visible=2025-07-18

  [4] Buscando partidos en pagina...
    [FOUND] Lokomotiv Moscow~Krasnodar | score: 1-1 | https://www.flashscore.com/match/AkfPxdhn/#/match-summary/match-summary
    [FOUND] Pari NN~Zenit | score: 0-2 | https://www.flashscore.com/match/neklXiMN/#/match-summary/match-summary
    [FOUND] Dynamo Moscow~Dynamo Makhachkala | score: 3-0 | https://www.flashscore.com/match/MwmNIh7U/#/match-summary/match-summary
    [FOUND] Akron Togliatti~Pari NN | score: 1-2 | https://www.flashscore.com/match/K8Vd0jjA/#/match-summary/match-summary
    [FOUND] Baltika~Spartak Moscow | score: 1-0 | https://www.flashscore.com/match/neXl2CLc/#/match

## Verificación — encontrados vs. base de datos
De los pendientes en DB, cuáles se **encontraron** en la página (por nombre+fecha) y cuáles **no**. Para los encontrados: score real de la página (reemplaza el `-1`) y si en DB están **sin estadísticas**.

In [5]:
assert 'found' in dir(), 'Corre la Etapa 3 primero (necesita `found`).'

stat_map = get_statistic_map([m['match_id'] for m in matches])
ids_found = set(found.keys())
encontrados    = [m for m in matches if m['match_id'] in ids_found]
no_encontrados = [m for m in matches if m['match_id'] not in ids_found]
n_sin_stats = sum(1 for m in encontrados if sin_stats(stat_map.get(m['match_id'])))

print(f'Pendientes DB: {len(matches)} | encontrados: {len(encontrados)} | NO encontrados: {len(no_encontrados)} | de los encontrados, sin stats: {n_sin_stats}')
print('\n-- ENCONTRADOS (coinciden con DB) --')
print('%-12s %-10s %-9s %s' % ('fecha', 'pág score', 'sin stats', 'partido'))
for m in encontrados:
    sc = found[m['match_id']]
    flag = 'sí' if sin_stats(stat_map.get(m['match_id'])) else 'no'
    print('%-12s %-10s %-9s %s' % (
        str(m['match_date']), f"{sc['home_result']}-{sc['visitor_result']}", flag, m['name']))
if no_encontrados:
    print('\n-- NO ENCONTRADOS en la página (revisar nombre/fecha o fuera de rango) --')
    for m in no_encontrados:
        print('  %-12s %s' % (str(m['match_date']), m['name']))
else:
    print('\nTodos los pendientes fueron encontrados en la página.')

Pendientes DB: 97 | encontrados: 97 | NO encontrados: 0 | de los encontrados, sin stats: 97

-- ENCONTRADOS (coinciden con DB) --
fecha        pág score  sin stats partido
2025-11-23   1-1        sí        Lokomotiv Moscow~Krasnodar
2025-11-23   0-2        sí        Pari NN~Zenit
2025-11-23   3-0        sí        Dynamo Moscow~Dynamo Makhachkala
2025-11-29   1-2        sí        Akron Togliatti~Pari NN
2025-11-29   1-0        sí        Baltika~Spartak Moscow
2025-11-29   2-0        sí        CSKA Moscow~Orenburg
2025-11-30   1-3        sí        FK Rostov~Lokomotiv Moscow
2025-11-30   1-0        sí        Zenit~Rubin Kazan
2025-11-30   5-0        sí        Krasnodar~Krylya Sovetov
2025-11-30   2-1        sí        Akhmat Grozny~Dynamo Moscow
2025-12-01   0-0        sí        Sochi~Dynamo Makhachkala
2025-12-05   1-0        sí        Akhmat Grozny~Orenburg
2025-12-06   1-1        sí        Spartak Moscow~Dynamo Moscow
2025-12-06   2-0        sí        FK Rostov~Rubin Kazan
2025-12-06   

## Opciones de actualización (las que irán al frontend)
- **MODO `'rapido'`** — solo resultados: `update_score` (por detalle) + `status → COMPLETED`. No navega al detalle, no toca `statistic`.
- **MODO `'completo'`** — score + estadísticas; escribe score + status + `statistic`.
- **SOLO_SIN_STATS `True`** — backfill de estadísticas sobre partidos **ya COMPLETED, con resultado ≠ -1 y sin `statistic`**. Fuerza modo `'completo'` y **no toca score ni status** (solo escribe `statistic`).

Regla: los `-1` siempre actualizan score; el modo busca-estadísticas solo opera sobre resultado ≠ -1 y status COMPLETED.

In [6]:
MODO = 'completo'           # 'rapido' | 'completo'
SOLO_SIN_STATS = True    # True -> backfill de stats (COMPLETED, score!=-1, sin statistic); fuerza 'completo'

if SOLO_SIN_STATS and MODO != 'completo':
    MODO = 'completo'
    print("[info] SOLO_SIN_STATS=True -> MODO forzado a 'completo'")
print(f'MODO = {MODO} | SOLO_SIN_STATS = {SOLO_SIN_STATS}')

MODO = completo | SOLO_SIN_STATS = True


## Población a procesar (según las opciones)
- **SOLO_SIN_STATS=False** → los pendientes de la Etapa 2/3 (`matches`/`found`): se completan y pasan a COMPLETED.
- **SOLO_SIN_STATS=True** → detector dedicado (COMPLETED, score≠-1, sin `statistic`); se cargan sus filas en la página para poder navegar a su detalle.

In [7]:
if SOLO_SIN_STATS:
    work_matches = get_stats_backfill_matches(LEAGUE_ID)
    es_backfill = True
    print(f'Backfill (COMPLETED, score!=-1, sin statistic): {len(work_matches)} matches')
    if work_matches:
        tgt = min(m['match_date'] for m in work_matches if m['match_date'])
        print(f'Cargando results hasta {tgt} para ubicarlos en la página...')
        load_until_date(driver, tgt)
        work_found = scan_results_page(driver, work_matches)
    else:
        work_found = {}
else:
    work_matches = matches
    work_found   = found
    es_backfill  = False

print(f'Población a procesar: {len(work_matches)} | mapeados en página: {len(work_found)} | backfill={es_backfill}')

Backfill (COMPLETED, score!=-1, sin statistic): 10 matches
Cargando results hasta 2026-03-21 para ubicarlos en la página...
  Cargando hasta: 2026-03-21
  [click 0] filas=279 | ultima fecha=2025-07-18
  Rango alcanzado.

  [4] Buscando partidos en pagina...
    [FOUND] Dynamo Makhachkala~FK Rostov | score: 1-2 | https://www.flashscore.com/match/Qysqg3JN/#/match-summary/match-summary
    [FOUND] CSKA Moscow~Zenit | score: 1-3 | https://www.flashscore.com/match/pIL9v44U/#/match-summary/match-summary
    [FOUND] Baltika~Rubin Kazan | score: 0-1 | https://www.flashscore.com/match/4MhzePlB/#/match-summary/match-summary
    [FOUND] FK Rostov~Orenburg | score: 0-1 | https://www.flashscore.com/match/ldVAXUjm/#/match-summary/match-summary
    [FOUND] Rubin Kazan~CSKA Moscow | score: 0-0 | https://www.flashscore.com/match/S2IxnU5t/#/match-summary/match-summary
    [FOUND] Krylya Sovetov~Lokomotiv Moscow | score: 2-0 | https://www.flashscore.com/match/hlSRTnjC/#/match-summary/match-summary
    [F

## Etapa 4 — Extraer y mostrar lo que se ESCRIBIRÍA en DB (dry-run)
Procesa `work_matches`/`work_found` según MODO. **No escribe.** Guard: en modo completo, si el score sigue en `-1` no se extraen estadísticas (resultado aún no confiable).

In [8]:
assert 'work_found' in dir(), 'Corre la celda de Población primero.'

objetivo = [m for m in work_matches if m['match_id'] in work_found]
filas = []
for m in objetivo:
    mid     = m['match_id']
    scraped = work_found[mid]
    home_r  = scraped['home_result']
    vis_r   = scraped['visitor_result']
    link    = scraped['link_details']
    dict_detail = get_math_details_ids(mid)
    score_por_detalle = {did: (home_r if hf else vis_r) for did, hf in dict_detail.items()}

    n_stats   = None  # rápido: no toca statistic
    statistic = None
    if MODO == 'completo':
        # Guard: solo extraer stats si el resultado es real (!= -1)
        if str(home_r) == '-1' or str(vis_r) == '-1':
            print(f'  [SKIP stats] {m["name"]}: score sigue -1, no se extraen estadísticas')
            n_stats = 0
        else:
            event_info = {'home': scraped.get('home',''), 'visitor': scraped.get('visitor','')}
            def _extract(drv, _l=link, _e=event_info):
                wait_load_details(drv, _l); get_match_info(drv, _e); return get_statistics_game(drv)
            statistic = '{}'
            try:
                r = retry_match(driver, link, _extract)
                if r: statistic = r
            except Exception as e:
                print(f'  [WARN] {m["name"]}: sin stats ({e})')
            try:    n_stats = len(eval(str(statistic)))
            except Exception: n_stats = 0
            # >>> VERIFICACIÓN: texto crudo extraído por get_statistics_game <<<
            print(f'  [TEXTO statistic] {m["name"]}  (url: {link})')
            print(f'      {statistic}')

    print(f'  [{MODO}] {m["name"]} | score {home_r}-{vis_r}'
          + ('' if n_stats is None else f' | stats={n_stats}'))
    filas.append({'fecha': m['match_date'], 'partido': m['name'],
                  'score': f'{home_r}-{vis_r}', 'n_stats': n_stats,
                  'statistic': statistic, 'detalles': score_por_detalle})

# ---- TABLA: lo que se CARGARÍA en DB (DRY-RUN) ----
status_lbl = 'permanece COMPLETED' if es_backfill else 'LIVE/SCHEDULED → COMPLETED'
print('\n' + '=' * 98)
print(f'ETAPA 4 — lo que se ESCRIBIRÍA en DB | MODO={MODO} SOLO_SIN_STATS={SOLO_SIN_STATS} (DRY-RUN)')
print('=' * 98)
print('%-12s %-8s %-7s %-26s %s' % ('fecha', 'score', 'stats', 'status', 'partido'))
print('-' * 98)
for f in filas:
    stats_col = '—' if f['n_stats'] is None else str(f['n_stats'])
    print('%-12s %-8s %-7s %-26s %s' % (
        str(f['fecha']), f['score'], stats_col, status_lbl, f['partido']))
print('-' * 98)
if es_backfill:
    print(f'Total: {len(filas)} | escritura real = UPDATE match.statistic  (score y status NO se tocan)')
elif MODO == 'rapido':
    print(f'Total: {len(filas)} | escritura real = update_score(points) por detalle + match.status=COMPLETED  (statistic NO se toca)')
else:
    print(f'Total: {len(filas)} | escritura real = update_score(points) por detalle + match.status=COMPLETED + UPDATE match.statistic')

if not es_backfill:
    print('\nScore por match_detail_id (lo que recibiría update_score):')
    for f in filas:
        print(f'  {f["partido"]:40s} -> {f["detalles"]}')

  [TEXTO statistic] Dynamo Makhachkala~FK Rostov  (url: https://www.flashscore.com/match/Qysqg3JN/#/match-summary/match-summary)
      {'Expected goals (xG)': {'home': '1.24', 'away': '0.15'}, 'Ball possession': {'home': '56%', 'away': '44%'}, 'Total shots': {'home': '9', 'away': '5'}, 'Shots on target': {'home': '2', 'away': '1'}, 'Big chances': {'home': '1', 'away': '0'}, 'Corner kicks': {'home': '3', 'away': '1'}, 'Passes': {'home': '67%(243/361)', 'away': '57%(159/280)'}, 'Yellow cards': {'home': '1', 'away': '4'}, 'xG on target (xGOT)': {'home': '1.39', 'away': '0.10'}, 'Shots off target': {'home': '3', 'away': '4'}, 'Blocked shots': {'home': '4', 'away': '0'}, 'Shots inside the box': {'home': '5', 'away': '3'}, 'Shots outside the box': {'home': '4', 'away': '2'}, 'Hit the woodwork': {'home': '0', 'away': '0'}, 'Touches in opposition box': {'home': '14', 'away': '11'}, 'Accurate through passes': {'home': '0', 'away': '0'}, 'Offsides': {'home': '6', 'away': '1'}, 'Free kicks': {'ho